# HDMI Capture Card Distraction Detection

Play a driving video on your phone or laptop screen and connect it to the board's HDMI capture card input via an HDMI cable. This notebook reads that live feed continuously and runs the same detection pipeline as the live-camera notebook, sourced from the capture card instead of the USB webcam.

**Speed note:** inference on this board takes roughly 9 seconds per frame (XNNPACK is disabled on this ARM32 build for safety). This is not a real-time video feed; the display below updates about once every 9 seconds. That is expected, not a bug.

**Alert logic:** uses `AlertHysteresis` imported directly from `alert_loop_infer.py` (already on this board at `/home/xilinx/alert_loop_infer.py`) - the exact same 8-consecutive-confident-tick logic already verified against `fpga/rtl/distraction_alert_controller.v`'s testbench. This notebook never re-implements that logic, so it can't drift out of sync with it. When a real alert fires (8 consecutive confident non-`safe_driving` ticks), it POSTs to the VM backend the same way the board's production alert loop does - check the Android app's history tab to see it arrive.

**How to stop:** use Jupyter's Interrupt Kernel (■) button at any time, or it stops on its own after `MAX_TICKS` frames (change the number below, or set to `None` to run until interrupted).


In [ ]:
import glob
import time
import numpy as np
import cv2
from tflite_runtime import interpreter as tflite
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output

from alert_loop_infer import AlertHysteresis, LABEL_NAMES, CONFIDENCE_THRESHOLD, post_alert, sound_buzzer_alert

MODEL_PATH = "/home/xilinx/mobilenetv2_crossview_finetuned_int8.tflite"
CAPTURE_CARD_USB_ID = "345f:2109"  # MacroSilicon MS2109 HDMI capture card
MAX_TICKS = 200  # safety limit (~30 min at ~9s/tick) - raise, or set to None to run until interrupted


def find_capture_device(usb_id):
    # /dev/videoN numbering shifts across reboots depending on USB enumeration
    # order - match by device name instead of trusting a fixed node.
    for name_path in sorted(glob.glob("/sys/class/video4linux/video*/name")):
        with open(name_path) as f:
            name = f.read().strip()
        if usb_id in name:
            dev_num = name_path.split("/")[-2]
            return f"/dev/{dev_num}"
    raise RuntimeError(f"No video device found matching '{usb_id}' - is the HDMI capture card connected?")


CAPTURE_DEVICE = find_capture_device(CAPTURE_CARD_USB_ID)
print(f"Using capture device: {CAPTURE_DEVICE}")

In [ ]:
interp = tflite.Interpreter(model_path=MODEL_PATH)
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("Model loaded:", MODEL_PATH)

In [ ]:
hysteresis = AlertHysteresis()
tick = 0

try:
    while MAX_TICKS is None or tick < MAX_TICKS:
        cap = cv2.VideoCapture(CAPTURE_DEVICE, cv2.CAP_V4L2)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            print("CAPTURE_FAILED - retrying in 1s")
            time.sleep(1)
            continue

        if frame.mean() < 1.0:
            print("WARNING: frame looks black - is the HDMI source actually connected and playing?")

        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
        x = resized.astype(inp['dtype'])[None, ...]

        t0 = time.time()
        interp.set_tensor(inp['index'], x)
        interp.invoke()
        y = interp.get_tensor(out['index'])[0]
        elapsed = time.time() - t0

        class_id = int(np.argmax(y))
        confidence = float(y[class_id])
        tick += 1

        fired = hysteresis.tick(class_id, confidence)
        alert_text = "ALERT ACTIVE" if hysteresis.alert else "monitoring"
        title = (
            f"tick {tick} | {LABEL_NAMES[class_id]} ({confidence:.0%}) | {elapsed:.1f}s/frame\n"
            f"distracted_run={hysteresis.distracted_run} safe_run={hysteresis.safe_run} | {alert_text}"
        )

        clear_output(wait=True)
        plt.figure(figsize=(6, 4.5))
        plt.imshow(img)
        plt.title(title, fontsize=10, color=("red" if hysteresis.alert else "black"))
        plt.axis("off")
        plt.show()

        if fired:
            print(f"*** ALERT FIRED: {LABEL_NAMES[class_id]} ({confidence:.0%}) - posting to backend ***")
            sound_buzzer_alert()
            post_alert(LABEL_NAMES[class_id], confidence)

except KeyboardInterrupt:
    print("Stopped by user.")

print(f"\nFinished after {tick} tick(s).")